# PARSeq-GeoAware — Colab Test Notebook
**Full inference pipeline: single image, batch, dataset evaluation**

> ⚠️ **Set runtime to T4 GPU first:** `Runtime → Change runtime type → T4 GPU`

Steps:
1. Clone repo from GitHub
2. Install dependencies
3. Mount Google Drive (optional — for your dataset)
4. Download Stage 3 checkpoint
5. Single image inference
6. Batch inference on a folder
7. Full dataset evaluation with metrics (WA / ±1 / CER / NED)
8. Visualise results

## Step 1 — Clone Repo

In [ ]:
import os

REPO_URL = 'https://github.com/Arni-123/PARSeq-GeoAware.git'
REPO_DIR = 'PARSeq-GeoAware'

![ -d "{REPO_DIR}" ] && (cd {REPO_DIR} && git pull) || git clone {REPO_URL}

%cd {REPO_DIR}
!git log --oneline -3
print('✅ Repo ready.')


## Step 2 — Install Dependencies

In [ ]:
!pip install -q timm opencv-python-headless scipy tqdm matplotlib editdistance gdown
print('✅ All dependencies installed.')

## Step 3 — Validate Repo Structure

In [ ]:
import os, ast

EXPECTED = [
    ('demo.py',              'Inference entry point'),
    ('models/model.py',      'PARSeqGeoAware architecture'),
    ('train.py',             'Training script'),
    ('evaluate.py',          'Evaluation script'),
    ('requirements.txt',     'Dependencies'),
]

print('=== File Check ===')
all_ok = True
for path, desc in EXPECTED:
    ok = os.path.exists(path)
    print(f'  {"✅" if ok else "❌ MISSING":12s} {path:30s} # {desc}')
    if not ok: all_ok = False

print()
if all_ok:
    print('✅ All required files present.')
else:
    print('⚠️  Some files missing — check repo.')

# Verify model symbols
REQUIRED = ['PARSeqGeoAware', 'set_charset', 'improved_ctc_decode', 'CHARSET', 'BLANK_IDX']
print('\n=== model.py Symbol Check ===')
if os.path.exists('models/model.py'):
    with open('models/model.py') as f:
        src = f.read()
    for sym in REQUIRED:
        ok = sym in src
        print(f'  {"✅" if ok else "❌ MISSING":12s} {sym}')

## Step 4 — Mount Google Drive *(optional)*
Skip this if you're uploading images directly.  
Mount if your dataset or checkpoint is on Drive.

In [ ]:
MOUNT_DRIVE = True   # ← set False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Drive mounted at /content/drive')
else:
    print('Skipped Drive mount.')

## Step 5 — Download Stage 3 Checkpoint
Downloads from Google Drive automatically.  
If `gdown` fails, download manually and upload via the Files panel (📁).

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)
CKPT    = 'checkpoints/stage3_best.pth'
FILE_ID = '1G6OBZN9h9FXj5iK_JckXAKvmacc1cI5T'

# ── Option A: Auto-download from public Google Drive link ─────────────────
!gdown "{FILE_ID}" -O "{CKPT}" --no-cookies

# ── Option B: Copy from your own Drive (uncomment if preferred) ───────────
# import shutil
# DRIVE_CKPT = '/content/drive/MyDrive/GeoAware_project/checkpoints_geoaware_v4/stage3_best.pth'
# if os.path.exists(DRIVE_CKPT):
#     shutil.copy(DRIVE_CKPT, CKPT)
#     print(f'Copied from Drive → {CKPT}')

# ── Verify ────────────────────────────────────────────────────────────────
size_mb = os.path.getsize(CKPT) / 1e6 if os.path.exists(CKPT) else 0
if size_mb > 10:
    print(f'✅ Checkpoint ready: {CKPT}  ({size_mb:.1f} MB)')
else:
    print('❌ Checkpoint not found or too small. Download manually:')
    print('   https://drive.google.com/file/d/1G6OBZN9h9FXj5iK_JckXAKvmacc1cI5T/view')
    print('   Then upload via Files panel (📁) → checkpoints/stage3_best.pth')


## Step 6 — Inspect Checkpoint

In [ ]:
import torch

CKPT = 'checkpoints/stage3_best.pth'
ckpt = torch.load(CKPT, map_location='cpu')

print('Top-level keys :', list(ckpt.keys()))
print('Stage          :', ckpt.get('stage', '?'))
print('Epoch          :', ckpt.get('epoch', '?'))
print('CTC loss       :', ckpt.get('ctc_loss', '?'))
print('Charset len    :', len(ckpt.get('charset', [])))
print('Config         :', ckpt.get('config', {}))

state_dict = ckpt.get('model_state_dict', ckpt)
print(f'\nTotal tensors  : {len(state_dict)}')

groups = {'encoder':0,'gfe':0,'fusion':0,'rectification':0,'head':0,'other':0}
for k in state_dict:
    matched = False
    for g in groups:
        if k.startswith(g):
            groups[g] += 1; matched = True; break
    if not matched:
        groups['other'] += 1

for g, n in groups.items():
    print(f'  {"✅" if n>0 else "❌":5s} {g:15s}: {n} tensors')

## Step 7 — Load Model

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from models.model import PARSeqGeoAware, set_charset, CHARSET, BLANK_IDX, improved_ctc_decode

CKPT   = 'checkpoints/stage3_best.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Read config from checkpoint so model always matches what was trained
ckpt = torch.load(CKPT, map_location=DEVICE)
cfg  = ckpt.get('config', {})

set_charset(36)   # 36 = standard a-z 0-9

model = PARSeqGeoAware(
    num_chars         = cfg.get('num_chars', len(CHARSET) + 1),
    use_geometric     = cfg.get('use_geometric',     True),
    use_rectification = cfg.get('use_rectification', True),
    use_tps           = cfg.get('use_tps',           True),
    use_attention     = False,
    max_len           = 25,
).to(DEVICE)

state_dict = ckpt.get('model_state_dict', ckpt)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model.eval()

print(f'✅ Model loaded  ({sum(p.numel() for p in model.parameters()):,} parameters)')
print(f'   Missing keys    : {len(missing)}')
print(f'   Unexpected keys : {len(unexpected)}')
if missing:    print(f'   {missing[:3]}')
if unexpected: print(f'   {unexpected[:3]}')

## Step 8 — Preprocessing & Predict Helper

In [ ]:
import torchvision.transforms as TF
from PIL import Image

# Must exactly match SimpleLineDataset transform in train__20_.py
TRANSFORM = TF.Compose([
    TF.Resize((64, 256)),
    TF.Grayscale(1),
    TF.ToTensor(),
    TF.Normalize(mean=[0.5], std=[0.5]),
])

def preprocess(image_path):
    """Load image → (1,1,64,256) tensor."""
    img = Image.open(image_path).convert('RGB')
    return TRANSFORM(img).unsqueeze(0)

def predict_image(image_path):
    """Full pipeline: path → predicted string."""
    tensor = preprocess(image_path).to(DEVICE)
    with torch.no_grad():
        try:
            log_probs, _ = model(tensor, return_features=True)
        except TypeError:
            out = model(tensor)
            if isinstance(out, (list, tuple)) and out[0].dim() == 2:
                log_probs = torch.stack(list(out[1:]), dim=0)
            else:
                log_probs = out[0]
        log_probs = log_probs.log_softmax(2)
        pred = improved_ctc_decode(log_probs)[0]
    return pred

print('✅ Helpers ready.')

## Step 9 — Single Image Inference
Upload one image and run inference.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

print('Upload a word-crop image (.jpg / .png):')
uploaded = files.upload()

for fname in uploaded:
    pred = predict_image(fname)
    print(f'\n  Image      : {fname}')
    print(f'  Prediction : "{pred}"')

    # Show image + prediction
    img = Image.open(fname).convert('RGB')
    plt.figure(figsize=(6, 3))
    plt.imshow(img)
    plt.title(f'Prediction: "{pred}"', fontsize=16,
              fontweight='bold', color='#27ae60', pad=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Step 10 — Batch Inference
Upload multiple images and see all predictions.

In [ ]:
from google.colab import files
import math, matplotlib.pyplot as plt

print('Upload multiple word-crop images:')
uploaded = files.upload()
image_paths = list(uploaded.keys())
print(f'\nUploaded {len(image_paths)} images')

# Run inference
results = []
for path in image_paths:
    pred = predict_image(path)
    results.append((path, pred))
    print(f'  {path:40s} → "{pred}"')

# Plot grid
n    = len(results)
cols = min(4, n)
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3 * rows))
axes = [axes] if n == 1 else axes.flatten() if rows > 1 else list(axes)

for ax, (path, pred) in zip(axes, results):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(f'"{pred}"', fontsize=12, fontweight='bold',
                 color='#27ae60', pad=6)
    ax.set_xlabel(os.path.basename(path), fontsize=8, color='gray')
    ax.axis('off')

# Hide unused axes
for ax in axes[n:]:
    ax.axis('off')

plt.suptitle('PARSeq-GeoAware — Batch Results', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('batch_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved → batch_results.png')

## Step 11 — Batch from Google Drive Folder
Point to a folder on your Drive that contains word-crop images.

In [ ]:
import os, math
import matplotlib.pyplot as plt

# ── Configure ──────────────────────────────────────────────────────────────
DRIVE_IMG_FOLDER = '/content/drive/MyDrive/GeoAware_project/test_images'
# ── Or use a local folder ──────────────────────────────────────────────────
# DRIVE_IMG_FOLDER = '/content/my_test_images'

if not os.path.isdir(DRIVE_IMG_FOLDER):
    print(f'❌ Folder not found: {DRIVE_IMG_FOLDER}')
    print('   Update DRIVE_IMG_FOLDER above or upload images in Step 10.')
else:
    exts = {'.jpg','.jpeg','.png','.bmp','.tiff','.webp'}
    image_paths = sorted([
        os.path.join(DRIVE_IMG_FOLDER, f)
        for f in os.listdir(DRIVE_IMG_FOLDER)
        if os.path.splitext(f)[1].lower() in exts
    ])
    print(f'Found {len(image_paths)} images in {DRIVE_IMG_FOLDER}')

    results = []
    for path in image_paths:
        pred = predict_image(path)
        results.append((path, pred))
        print(f'  {os.path.basename(path):35s} → "{pred}"')

    # Save to txt
    out_txt = 'drive_batch_predictions.txt'
    with open(out_txt, 'w') as f:
        for path, pred in results:
            f.write(f'{path}\t{pred}\n')
    print(f'\n✅ Saved {len(results)} predictions → {out_txt}')

## Step 12 — Full Dataset Evaluation with Metrics
Evaluate on a ground-truth `.txt` file (tab-separated: `image_path\tlabel`).

Metrics reported:
- **WA** — Word Accuracy (exact match)
- **±1 Acc** — Tolerates 1 insertion/deletion/substitution
- **CER** — Character Error Rate (global normalization)
- **NED** — Normalized Edit Distance (per-word average)

In [ ]:
import os, torch, editdistance
from tqdm import tqdm

# ── Configure ──────────────────────────────────────────────────────────────
# Point to any of your test gt files, e.g.:
GT_TXT = '/content/data/test/art_test_gt.txt'
# GT_TXT = '/content/data/test/iiit5k_test.txt'
# GT_TXT = '/content/data/test/totaltext_test_gt.txt'

MAX_SAMPLES = None   # None = evaluate all; set e.g. 500 for quick test

CHARSET_SET = set(CHARSET)   # for OOV filtering

# ── Load ground truth ─────────────────────────────────────────────────────
if not os.path.exists(GT_TXT):
    print(f'❌ GT file not found: {GT_TXT}')
    print('   Update GT_TXT above or mount Drive in Step 4.')
else:
    samples = []
    with open(GT_TXT) as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line:
                continue
            path, gt = line.split('\t', 1)
            gt_clean = ''.join(c for c in gt.strip().lower() if c in CHARSET_SET)
            if gt_clean and os.path.exists(path):
                samples.append((path, gt_clean))
            if MAX_SAMPLES and len(samples) >= MAX_SAMPLES:
                break

    print(f'Evaluating {len(samples)} samples from {os.path.basename(GT_TXT)}...')

    # ── Evaluation loop ───────────────────────────────────────────────────
    n_exact = n_pm1 = n_pm2 = 0
    sum_ed = sum_gt_len = 0
    sum_ned = 0.0
    errors = []

    for path, gt in tqdm(samples):
        try:
            pred = predict_image(path)
        except Exception as e:
            pred = ''

        ed = editdistance.eval(pred, gt)

        if pred == gt:               n_exact += 1
        if ed <= 1:                  n_pm1   += 1
        if ed <= 2:                  n_pm2   += 1

        sum_ed     += ed
        sum_gt_len += len(gt)
        sum_ned    += 1.0 - ed / max(len(pred), len(gt), 1)

        if pred != gt:
            errors.append((os.path.basename(path), gt, pred, ed))

    N   = len(samples)
    WA  = 100 * n_exact / N
    PM1 = 100 * n_pm1   / N
    PM2 = 100 * n_pm2   / N
    CER = 100 * sum_ed  / max(sum_gt_len, 1)
    NED = sum_ned / N

    print()
    print('=' * 50)
    print(f'  Dataset : {os.path.basename(GT_TXT)}')
    print(f'  Samples : {N}')
    print('=' * 50)
    print(f'  Word Accuracy (WA)    : {WA:.2f}%')
    print(f'  ±1 Char Accuracy      : {PM1:.2f}%')
    print(f'  ±2 Char Accuracy      : {PM2:.2f}%')
    print(f'  Char Error Rate (CER) : {CER:.2f}%')
    print(f'  Norm. Edit Dist (NED) : {NED:.4f}')
    print('=' * 50)

    # Show top-10 errors
    errors.sort(key=lambda x: -x[3])
    print(f'\nTop-10 worst errors (highest edit distance):')
    print(f'  {"File":30s} {"GT":15s} {"Pred":15s} ED')
    print(f'  {"-"*65}')
    for fname, gt, pred, ed in errors[:10]:
        print(f'  {fname:30s} {gt:15s} {pred:15s} {ed}')

## Step 13 — Multi-Dataset Evaluation Table

In [ ]:
import os, editdistance
from tqdm import tqdm

# ── Configure all test sets ────────────────────────────────────────────────
TEST_SETS = {
    'IIIT5K'    : '/content/data/test/iiit5k_test.txt',
    'ArT'       : '/content/data/test/art_test_gt.txt',
    'Total-Text': '/content/data/test/totaltext_test_gt.txt',
    # Add more as needed:
    # 'SVT'     : '/content/data/test/svt_test.txt',
    # 'ICDAR13' : '/content/data/test/icdar13_test.txt',
    # 'ICDAR15' : '/content/data/test/icdar15_test.txt',
}

CHARSET_SET = set(CHARSET)
all_results = {}

for name, gt_txt in TEST_SETS.items():
    if not os.path.exists(gt_txt):
        print(f'  ⚠  {name}: file not found → {gt_txt}')
        continue

    samples = []
    with open(gt_txt) as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line: continue
            path, gt = line.split('\t', 1)
            gt_clean = ''.join(c for c in gt.strip().lower() if c in CHARSET_SET)
            if gt_clean and os.path.exists(path):
                samples.append((path, gt_clean))

    if not samples:
        print(f'  ⚠  {name}: no valid samples')
        continue

    n_exact = n_pm1 = 0
    sum_ed = sum_gt_len = 0
    sum_ned = 0.0

    for path, gt in tqdm(samples, desc=name, ncols=70):
        try:    pred = predict_image(path)
        except: pred = ''
        ed = editdistance.eval(pred, gt)
        if pred == gt: n_exact += 1
        if ed <= 1:    n_pm1   += 1
        sum_ed     += ed
        sum_gt_len += len(gt)
        sum_ned    += 1.0 - ed / max(len(pred), len(gt), 1)

    N = len(samples)
    all_results[name] = {
        'N'  : N,
        'WA' : 100 * n_exact / N,
        'PM1': 100 * n_pm1   / N,
        'CER': 100 * sum_ed  / max(sum_gt_len, 1),
        'NED': sum_ned / N,
    }

# ── Print table ───────────────────────────────────────────────────────────
print()
print('=' * 72)
print(f'  {"Dataset":15s} {"N":>7s}  {"WA":>7s}  {"±1 Acc":>7s}  {"CER":>7s}  {"NED":>7s}')
print(f'  {"-"*67}')
for name, r in all_results.items():
    print(f'  {name:15s} {r["N"]:>7,}  {r["WA"]:>6.2f}%  '
          f'{r["PM1"]:>6.2f}%  {r["CER"]:>6.2f}%  {r["NED"]:>7.4f}')
print('=' * 72)

## Step 14 — Run via demo.py CLI
Uses the committed `demo.py` exactly as a user would from the command line.

In [ ]:
# ── Single image via demo.py ──────────────────────────────────────────────
# Replace gt_0_5.jpg with your image path
!python demo.py --checkpoint checkpoints/stage3_best.pth --image gt_0_5.jpg --charset 36


In [ ]:
# Folder via demo.py — update FOLDER below then run
import subprocess, os

FOLDER = '/content/my_test_images'   # <- change this to your folder

if not os.path.isdir(FOLDER):
    print(f'❌ Folder not found: {FOLDER}')
    print('   Update FOLDER above.')
else:
    cmd = [
        'python', 'demo.py',
        '--checkpoint',   'checkpoints/stage3_best.pth',
        '--image_folder', FOLDER,
        '--output',       'predictions.txt',
        '--charset',      '36',
        '--visualise',
    ]
    subprocess.run(cmd)


In [ ]:
# ── Show predictions.txt ─────────────────────────────────────────────────
if os.path.exists('predictions.txt'):
    with open('predictions.txt') as f:
        lines = f.readlines()
    print(f'{len(lines)} predictions:')
    for line in lines[:20]:
        path, pred = line.strip().split('\t', 1)
        print(f'  {os.path.basename(path):35s} → "{pred}"')
    if len(lines) > 20:
        print(f'  ... and {len(lines)-20} more')

## Step 15 — Ablation: Component Contribution
Disable components at inference time and measure accuracy drop.
Replicates Table 9 from the paper.

In [ ]:
import editdistance
from tqdm import tqdm

GT_TXT    = '/content/data/test/art_test_gt.txt'   # ← change as needed
MAX_EVAL  = 500   # quick ablation; set None for full

CHARSET_SET = set(CHARSET)

if not os.path.exists(GT_TXT):
    print(f'❌ {GT_TXT} not found')
else:
    # Load samples
    samples = []
    with open(GT_TXT) as f:
        for line in f:
            line = line.strip()
            if not line or '\t' not in line: continue
            path, gt = line.split('\t', 1)
            gt_clean = ''.join(c for c in gt.strip().lower() if c in CHARSET_SET)
            if gt_clean and os.path.exists(path):
                samples.append((path, gt_clean))
            if MAX_EVAL and len(samples) >= MAX_EVAL: break

    def eval_config(use_geo, use_rect, use_tps, label):
        model.use_geometric     = use_geo
        model.use_rectification = use_rect
        model.use_tps           = use_tps
        n_exact = 0
        for path, gt in tqdm(samples, desc=label, ncols=70, leave=False):
            try:    pred = predict_image(path)
            except: pred = ''
            if pred == gt: n_exact += 1
        wa = 100 * n_exact / len(samples)
        return wa

    configs = [
        (False, False, False, 'ViT+CTC only (baseline)'),
        (True,  False, False, '+GFE (no fusion)'),
        (True,  True,  False, '+GFE + Fusion + Affine rect'),
        (True,  True,  True,  'Full model (affine+TPS) ★'),
    ]

    print(f'\nAblation on {len(samples)} samples from {os.path.basename(GT_TXT)}')
    print('=' * 55)
    print(f'  {"Configuration":35s}  WA (%)')
    print(f'  {"-"*50}')
    baseline = None
    for use_geo, use_rect, use_tps, label in configs:
        wa = eval_config(use_geo, use_rect, use_tps, label)
        delta = f'  (+{wa-baseline:.2f}pp)' if baseline is not None else ''
        print(f'  {label:35s}  {wa:.2f}%{delta}')
        if baseline is None: baseline = wa

    # Restore full model
    model.use_geometric = model.use_rectification = model.use_tps = True
    print('=' * 55)
    print('✅ Model restored to full configuration.')

## Step 16 — Inference Timing (replicates Table 4)

In [ ]:
import torch, time

WARMUP = 100
TIMED  = 500
dummy  = torch.zeros(1, 1, 64, 256).to(DEVICE)

configs = [
    (False, False, False, 'ViT+CTC only'),
    (True,  False, False, '+GFE'),
    (True,  True,  False, '+GFE + Affine'),
    (True,  True,  True,  'Full model (aff+TPS) ★'),
]

print(f'Timing on {DEVICE} — {WARMUP} warmup + {TIMED} timed runs (batch=1, FP32)')
print('=' * 58)
print(f'  {"Configuration":30s}  {"Mean(ms)":>9s}  {"FPS":>6s}')
print(f'  {"-"*53}')

model.eval()
for use_geo, use_rect, use_tps, label in configs:
    model.use_geometric     = use_geo
    model.use_rectification = use_rect
    model.use_tps           = use_tps

    # Warmup
    with torch.no_grad():
        for _ in range(WARMUP):
            try:    _ = model(dummy, return_features=True)
            except: _ = model(dummy)

    # Timed
    times = []
    with torch.no_grad():
        for _ in range(TIMED):
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            t0 = time.perf_counter()
            try:    _ = model(dummy, return_features=True)
            except: _ = model(dummy)
            if DEVICE.type == 'cuda': torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)

    import statistics
    mean_ms = statistics.mean(times)
    fps     = 1000 / mean_ms
    print(f'  {label:30s}  {mean_ms:>8.1f}ms  {fps:>5.0f}')

# Restore
model.use_geometric = model.use_rectification = model.use_tps = True
print('=' * 58)

## Step 17 — Download All Results

In [ ]:
import shutil, os
from google.colab import files

# Bundle everything worth keeping
outputs = [
    'predictions.txt',
    'batch_results.png',
    'drive_batch_predictions.txt',
]

for f in outputs:
    if os.path.exists(f):
        files.download(f)
        print(f'  Downloaded: {f}')
    else:
        print(f'  Not found (skipped): {f}')